# Random Forest Model Optimisation

Main choices:

1. Hyperparameters are selected using cross-validation AUPRC on the model-training set.
2. The probability threshold is selected only on the validation set.
3. The final test set is used once, after both the model settings and threshold have been fixed.
4. The target validation recall is 80%, matching the current project requirement.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


## 1. Load the cleaned dataset and create the output folder

In [2]:
PROJECT_ROOT = Path.cwd()

for parent in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
    candidate = (
        parent
        / "Processed_Dataset"
        / "diabetic_data_cleaned_stage1.csv"
    )

    if candidate.exists():
        DATA_PATH = candidate
        PROJECT_ROOT = parent
        break
else:
    raise FileNotFoundError(
        "Could not find "
        "Processed_Dataset/diabetic_data_cleaned_stage1.csv"
    )

OUTPUT_DIR = (
    PROJECT_ROOT
    / "Model_Results"
    / "random_forest_optimisation"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

df = pd.read_csv(DATA_PATH)

print("Dataset path:")
print(DATA_PATH)

print("\nDataset shape:")
print(df.shape)

df.head()


Dataset path:
/Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Processed_Dataset/diabetic_data_cleaned_stage1.csv

Dataset shape:
(69987, 56)


,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,...,diabetesMed,readmitted,readmitted_30,hba1c_group,primary_diagnosis,age_group,discharge_group,race_group,admission_source_group,medical_specialty_group
0,24437208,135,Caucasian,Female,[50-60),2,1,1,8,Cardiology,...,Yes,<30,1,No test was performed,Circulatory,30-60,Home,Caucasian,Physician/clinic referral,Cardiology
1,29758806,378,Caucasian,Female,[50-60),3,1,1,2,Surgery-Neuro,...,No,NO,0,No test was performed,Musculoskeletal,30-60,Home,Caucasian,Physician/clinic referral,Surgery
2,189899286,729,Caucasian,Female,[80-90),1,3,7,4,InternalMedicine,...,Yes,NO,0,Normal result of the test,Injury,>60,Other,Caucasian,Emergency room,Internal Medicine
3,64331490,774,Caucasian,Female,[80-90),1,1,7,3,InternalMedicine,...,Yes,NO,0,"High, medication changed",Other,>60,Home,Caucasian,Emergency room,Internal Medicine
4,14824206,927,AfricanAmerican,Female,[30-40),1,1,7,5,InternalMedicine,...,Yes,NO,0,No test was performed,Genitourinary,30-60,Home,AfricanAmerican,Emergency room,Internal Medicine


## 2. Select the same modelling features used by the other models

In [3]:
target_col = "readmitted_30"

categorical_features = [
    "gender",
    "race_group",
    "age_group",
    "admission_source_group",
    "discharge_group",
    "medical_specialty_group",
    "primary_diagnosis",
    "hba1c_group",
    "max_glu_serum",
    "diabetesMed"
]

numeric_features = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

model_features = (
    categorical_features
    + numeric_features
)

missing_features = [
    feature
    for feature in model_features
    if feature not in df.columns
]

if missing_features:
    raise ValueError(
        "These modelling features are missing: "
        f"{missing_features}"
    )

X = df[model_features].copy()
y = df[target_col].astype(int).copy()

print("X shape:")
print(X.shape)

print("\nFeatures used:")
print(X.columns.tolist())

print("\nTarget counts:")
print(y.value_counts())

print("\nTarget proportions:")
print(y.value_counts(normalize=True))


X shape:
(69987, 18)

Features used:
['gender', 'race_group', 'age_group', 'admission_source_group', 'discharge_group', 'medical_specialty_group', 'primary_diagnosis', 'hba1c_group', 'max_glu_serum', 'diabetesMed', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']

Target counts:
readmitted_30
0    63702
1     6285
Name: count, dtype: int64

Target proportions:
readmitted_30
0    0.910198
1    0.089802
Name: proportion, dtype: float64


In [4]:
forbidden_features = {
    "race",
    "age",
    "medical_specialty",
    "diag_1",
    "A1Cresult",
    "admission_type_id",
    "admission_source_id",
    "discharge_disposition_id",
    "readmitted",
    "readmitted_30",
    "encounter_id",
    "patient_nbr"
}

unexpected_features = (
    forbidden_features
    .intersection(X.columns)
)

assert not unexpected_features, (
    "Unexpected or potentially leaking features found: "
    f"{unexpected_features}"
)

assert not X.columns.duplicated().any(), (
    "Duplicate column names were found in X."
)

assert len(X) == len(y), (
    "X and y contain different numbers of rows."
)

assert y.isna().sum() == 0, (
    "The target contains missing values."
)

assert set(y.unique()).issubset({0, 1}), (
    "The target must contain only 0 and 1."
)

print("Feature and target checks passed.")


Feature and target checks passed.


## 3. Create model-training, validation, and final-test sets

The final test set must remain untouched until the model settings and threshold have been selected.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

X_model_train, X_val, y_model_train, y_val = (
    train_test_split(
        X_train,
        y_train,
        test_size=0.25,
        stratify=y_train,
        random_state=42
    )
)

print("Model-training set:")
print(X_model_train.shape)
print(y_model_train.value_counts())
print(y_model_train.value_counts(normalize=True))

print("\nValidation set:")
print(X_val.shape)
print(y_val.value_counts())
print(y_val.value_counts(normalize=True))

print("\nFinal test set:")
print(X_test.shape)
print(y_test.value_counts())
print(y_test.value_counts(normalize=True))


Model-training set:
(41991, 18)
readmitted_30
0    38220
1     3771
Name: count, dtype: int64
readmitted_30
0    0.910195
1    0.089805
Name: proportion, dtype: float64

Validation set:
(13998, 18)
readmitted_30
0    12741
1     1257
Name: count, dtype: int64
readmitted_30
0    0.910201
1    0.089799
Name: proportion, dtype: float64

Final test set:
(13998, 18)
readmitted_30
0    12741
1     1257
Name: count, dtype: int64
readmitted_30
0    0.910201
1    0.089799
Name: proportion, dtype: float64


## 4. Preprocessing

Random Forest does not require feature scaling. Categorical variables are imputed and one-hot encoded. Numeric variables are median-imputed.

In [6]:
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]
)

random_forest_preprocess = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_transformer,
            categorical_features
        ),
        (
            "numeric",
            numeric_transformer,
            numeric_features
        )
    ],
    remainder="drop"
)

print("Random Forest preprocessing created.")


Random Forest preprocessing created.


## 5. Evaluation and threshold-selection helper functions

In [7]:
def evaluate_predictions_from_proba(
    y_true,
    y_proba,
    threshold=0.5,
    model_name="Model"
):
    """Evaluate binary predictions created from probabilities."""

    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)

    y_pred = (
        y_proba >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    total = tn + fp + fn + tp
    actual_positive = tp + fn
    actual_negative = tn + fp
    predicted_positive = tp + fp
    predicted_negative = tn + fn

    specificity = (
        tn / actual_negative
        if actual_negative > 0
        else np.nan
    )

    false_positive_rate = (
        fp / actual_negative
        if actual_negative > 0
        else np.nan
    )

    false_negative_rate = (
        fn / actual_positive
        if actual_positive > 0
        else np.nan
    )

    predicted_positive_rate = (
        predicted_positive / total
        if total > 0
        else np.nan
    )

    patients_flagged_per_true_readmission = (
        predicted_positive / tp
        if tp > 0
        else np.nan
    )

    return {
        "model": model_name,
        "threshold": float(threshold),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "specificity": specificity,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "f2": fbeta_score(
            y_true,
            y_pred,
            beta=2,
            zero_division=0
        ),
        "auroc": roc_auc_score(
            y_true,
            y_proba
        ),
        "auprc": average_precision_score(
            y_true,
            y_proba
        ),
        "brier_score": brier_score_loss(
            y_true,
            y_proba
        ),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
        "predicted_positive": int(predicted_positive),
        "predicted_negative": int(predicted_negative),
        "predicted_positive_rate": predicted_positive_rate,
        "patients_flagged_per_true_readmission_found": (
            patients_flagged_per_true_readmission
        )
    }


In [8]:
def confusion_matrix_from_proba(
    y_true,
    y_proba,
    threshold=0.5
):
    """Create a labelled confusion matrix from probabilities."""

    y_pred = (
        np.asarray(y_proba) >= threshold
    ).astype(int)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    return pd.DataFrame(
        cm,
        index=[
            "Actual not readmitted",
            "Actual readmitted"
        ],
        columns=[
            "Predicted not readmitted",
            "Predicted readmitted"
        ]
    )


In [9]:
def threshold_sweep(
    y_true,
    y_proba,
    model_name="Model",
    thresholds=None
):
    """Calculate performance across probability thresholds."""

    if thresholds is None:
        thresholds = np.round(
            np.arange(
                0.01,
                0.951,
                0.01
            ),
            2
        )

    results = [
        evaluate_predictions_from_proba(
            y_true=y_true,
            y_proba=y_proba,
            threshold=threshold,
            model_name=model_name
        )
        for threshold in thresholds
    ]

    return pd.DataFrame(results)


In [10]:
def choose_threshold_for_minimum_recall(
    y_true,
    y_proba,
    min_recall=0.80,
    model_name="Model"
):
    """
    Select the threshold with the lowest false-positive rate
    among thresholds that achieve the required recall.
    """

    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)

    false_positive_rates, recalls, thresholds = roc_curve(
        y_true,
        y_proba,
        drop_intermediate=False
    )

    candidate_table = pd.DataFrame({
        "threshold": thresholds,
        "recall": recalls,
        "false_positive_rate": false_positive_rates,
        "specificity": 1 - false_positive_rates
    })

    candidate_table = candidate_table[
        np.isfinite(
            candidate_table["threshold"]
        )
    ].copy()

    eligible_candidates = candidate_table[
        candidate_table["recall"] >= min_recall
    ].copy()

    if eligible_candidates.empty:
        raise ValueError(
            "No threshold achieved recall >= "
            f"{min_recall:.2f}."
        )

    eligible_candidates = (
        eligible_candidates
        .sort_values(
            by=[
                "false_positive_rate",
                "threshold"
            ],
            ascending=[
                True,
                False
            ]
        )
        .reset_index(drop=True)
    )

    selected_threshold = float(
        eligible_candidates.iloc[0]["threshold"]
    )

    selected_metrics = evaluate_predictions_from_proba(
        y_true=y_true,
        y_proba=y_proba,
        threshold=selected_threshold,
        model_name=model_name
    )

    return (
        selected_threshold,
        selected_metrics,
        eligible_candidates
    )


## 6. Cross-validation settings and hyperparameter search

AUPRC is used for model selection because only about 9% of encounters are positive. Recall is not used directly inside the search because recall depends strongly on the probability threshold. The threshold is selected later on the validation set.

The Random Forest itself uses one CPU during each fit, while RandomizedSearchCV parallelises the separate fits. This avoids nested parallelism and excessive memory use.

In [11]:
RECALL_TARGET = 0.80
SEARCH_ITERATIONS = 60

cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print(
    f"Required validation recall: "
    f"{RECALL_TARGET:.0%}"
)

print(
    "Cross-validation folds:",
    cross_validation.n_splits
)

print(
    "Random search candidates:",
    SEARCH_ITERATIONS
)


Required validation recall: 80%
Cross-validation folds: 5
Random search candidates: 60


In [12]:
random_forest_param_distributions = {
    "model__n_estimators": [
        200,
        300,
        500,
        700
    ],

    "model__criterion": [
        "gini",
        "entropy",
        "log_loss"
    ],

    "model__max_depth": [
        6,
        8,
        10,
        12,
        16,
        20,
        None
    ],

    "model__min_samples_split": [
        2,
        10,
        25,
        50,
        100,
        200
    ],

    "model__min_samples_leaf": [
        1,
        5,
        10,
        25,
        50,
        100,
        200
    ],

    "model__max_features": [
        "sqrt",
        "log2",
        0.30,
        0.50,
        0.75,
        None
    ],

    "model__max_samples": [
        0.50,
        0.75,
        None
    ],

    "model__class_weight": [
        None,
        "balanced",
        "balanced_subsample",
        {0: 1, 1: 2},
        {0: 1, 1: 4},
        {0: 1, 1: 6},
        {0: 1, 1: 8}
    ]
}


In [13]:
random_forest_pipeline = Pipeline(
    steps=[
        (
            "preprocess",
            random_forest_preprocess
        ),
        (
            "model",
            RandomForestClassifier(
                random_state=42,
                bootstrap=True,
                n_jobs=1
            )
        )
    ]
)

random_forest_pipeline


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default

In [14]:
random_forest_search = RandomizedSearchCV(
    estimator=random_forest_pipeline,

    param_distributions=(
        random_forest_param_distributions
    ),

    n_iter=SEARCH_ITERATIONS,

    scoring={
        "auprc": "average_precision",
        "auroc": "roc_auc"
    },

    refit="auprc",

    cv=cross_validation,

    n_jobs=-1,

    verbose=1,

    random_state=42,

    return_train_score=True,

    error_score="raise"
)

random_forest_search.fit(
    X_model_train,
    y_model_train
)


Fitting 5 folds for each of 60 candidates, totalling 300 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__class_weight': [None, 'balanced', ...], 'model__criterion': ['gini', 'entropy', ...], 'model__max_depth': [6, 8, ...], 'model__max_features': ['sqrt', 'log2', ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",60
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.","{'auprc': 'average_precision', 'auroc': 'roc_auc'}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'auprc'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold

## 7. Inspect the selected model and the strongest search candidates

In [15]:
print("Best Random Forest parameters:")

for parameter, value in (
    random_forest_search
    .best_params_
    .items()
):
    print(f"{parameter}: {value}")

print("\nBest cross-validation AUPRC:")
print(random_forest_search.best_score_)


Best Random Forest parameters:
model__n_estimators: 500
model__min_samples_split: 2
model__min_samples_leaf: 5
model__max_samples: None
model__max_features: log2
model__max_depth: 8
model__criterion: gini
model__class_weight: None

Best cross-validation AUPRC:
0.1520746516245111


In [16]:
random_forest_cv_results = pd.DataFrame(
    random_forest_search.cv_results_
)

random_forest_cv_results_selected = (
    random_forest_cv_results[
        [
            "rank_test_auprc",
            "mean_test_auprc",
            "std_test_auprc",
            "mean_train_auprc",
            "std_train_auprc",
            "mean_test_auroc",
            "std_test_auroc",
            "param_model__n_estimators",
            "param_model__criterion",
            "param_model__max_depth",
            "param_model__min_samples_split",
            "param_model__min_samples_leaf",
            "param_model__max_features",
            "param_model__max_samples",
            "param_model__class_weight"
        ]
    ]
    .sort_values("rank_test_auprc")
    .reset_index(drop=True)
)

random_forest_cv_results_selected[
    "train_validation_auprc_gap"
] = (
    random_forest_cv_results_selected[
        "mean_train_auprc"
    ]
    -
    random_forest_cv_results_selected[
        "mean_test_auprc"
    ]
)

random_forest_cv_results_selected.to_csv(
    OUTPUT_DIR
    / "random_forest_cv_results.csv",
    index=False
)

random_forest_cv_results_selected.head(20)


,rank_test_auprc,mean_test_auprc,std_test_auprc,mean_train_auprc,std_train_auprc,mean_test_auroc,std_test_auroc,param_model__n_estimators,param_model__criterion,param_model__max_depth,param_model__min_samples_split,param_model__min_samples_leaf,param_model__max_features,param_model__max_samples,param_model__class_weight,train_validation_auprc_gap
0,1,0.152075,0.006281,0.256461,0.003251,0.636290,0.007688,500,gini,8,2,5,log2,None,None,0.104387
1,2,0.151448,0.004635,0.212771,0.001070,0.637670,0.005705,500,log_loss,16,200,50,sqrt,None,None,0.061323
2,3,0.151006,0.005499,0.179112,0.001278,0.632471,0.006344,300,gini,6,200,50,None,None,balanced_subsample,0.028106
3,4,0.150797,0.005348,0.507767,0.005634,0.635644,0.005326,700,entropy,None,25,5,log2,0.5,balanced,0.356970
4,5,0.150639,0.005272,0.230027,0.002665,0.632789,0.005748,700,log_loss,16,200,25,0.5,None,balanced,0.079388
5,6,0.150461,0.005120,0.251440,0.002798,0.633689,0.005342,500,entropy,None,2,50,0.5,0.75,None,0.100979
6,7,0.150204,0.004626,0.226194,0.002826,0.633157,0.005711,700,gini,8,2,25,0.5,0.75,"{0: 1, 1: 4}",0.075990
7,8,0.150146,0.005140,0.235613,0.003010,0.632585,0.005271,200,gini,16,50,50,0.5,0.5,"{0: 1, 1: 6}",0.085467
8,9,0.150031,0.005129,0.259092,0.003653,0.632662,0.005322,500,entropy,None,50,50,0.75,0.75,None,0.109061
9,10,0.150015,0.004176,0.189937,0.001898,0.635850,0.005903,700,entropy,12,50,100,0.3,0.75,None,0.039922


In [17]:
best_random_forest_model = (
    random_forest_search
    .best_estimator_
)

# Parallelise predictions and later calculations after the search.
best_random_forest_model.named_steps[
    "model"
].set_params(n_jobs=-1)

best_random_forest_model


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](18,)","['gender','race_group','age_group',...,'number_emergency', 'number_inpatient','number_diagnoses']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,18
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifyi

In [18]:
fitted_random_forest = (
    best_random_forest_model
    .named_steps["model"]
)

individual_tree_depths = np.array([
    tree.get_depth()
    for tree in fitted_random_forest.estimators_
])

individual_tree_leaves = np.array([
    tree.get_n_leaves()
    for tree in fitted_random_forest.estimators_
])

forest_structure_summary = pd.Series({
    "number_of_trees": len(
        fitted_random_forest.estimators_
    ),
    "minimum_tree_depth": individual_tree_depths.min(),
    "median_tree_depth": np.median(
        individual_tree_depths
    ),
    "mean_tree_depth": individual_tree_depths.mean(),
    "maximum_tree_depth": individual_tree_depths.max(),
    "minimum_leaf_count": individual_tree_leaves.min(),
    "median_leaf_count": np.median(
        individual_tree_leaves
    ),
    "mean_leaf_count": individual_tree_leaves.mean(),
    "maximum_leaf_count": individual_tree_leaves.max()
})

forest_structure_summary.to_csv(
    OUTPUT_DIR
    / "random_forest_structure_summary.csv",
    header=["value"]
)

forest_structure_summary


number_of_trees       500.000
minimum_tree_depth      8.000
median_tree_depth       8.000
mean_tree_depth         8.000
maximum_tree_depth      8.000
minimum_leaf_count     72.000
median_leaf_count     138.000
mean_leaf_count       138.124
maximum_leaf_count    192.000
dtype: float64

## 8. Evaluate the selected model on the validation set at the default threshold

In [19]:
y_val_proba_random_forest = (
    best_random_forest_model
    .predict_proba(X_val)[:, 1]
)

print(
    "Minimum validation probability:",
    y_val_proba_random_forest.min()
)

print(
    "Maximum validation probability:",
    y_val_proba_random_forest.max()
)

print(
    "Number of unique validation probabilities:",
    np.unique(y_val_proba_random_forest).size
)


Minimum validation probability: 0.03424425873852267
Maximum validation probability: 0.2404306402196879
Number of unique validation probabilities: 13996


In [20]:
random_forest_probability_summary = (
    pd.DataFrame({
        "actual_class": np.asarray(y_val),
        "predicted_readmission_probability": (
            y_val_proba_random_forest
        )
    })
    .groupby("actual_class")
    ["predicted_readmission_probability"]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90
        ]
    )
)

random_forest_probability_summary.to_csv(
    OUTPUT_DIR
    / "random_forest_validation_probability_summary.csv"
)

random_forest_probability_summary


,count,mean,std,min,10%,25%,50%,75%,90%,max
actual_class,,,,,,,,,,
0,12741.0,0.088572,0.029448,0.034244,0.055967,0.065363,0.080160,0.113134,0.126623,0.240431
1,1257.0,0.101961,0.032909,0.037542,0.063167,0.073860,0.101954,0.123780,0.136136,0.225922


In [21]:
random_forest_val_default_results = (
    evaluate_predictions_from_proba(
        y_true=y_val,
        y_proba=y_val_proba_random_forest,
        threshold=0.5,
        model_name=(
            "Random Forest validation default"
        )
    )
)

pd.Series(
    random_forest_val_default_results
)


model                                          Random Forest validation default
threshold                                                                   0.5
accuracy                                                               0.910201
precision                                                                   0.0
recall                                                                      0.0
specificity                                                                 1.0
false_positive_rate                                                         0.0
false_negative_rate                                                         1.0
f1                                                                          0.0
f2                                                                          0.0
auroc                                                                  0.622744
auprc                                                                   0.13961
brier_score                             

## 9. Select the validation threshold that reaches at least 80% recall

Among all validation thresholds that achieve the recall target, this rule selects the one with the lowest false-positive rate.

In [22]:
(
    random_forest_selected_threshold,
    random_forest_val_selected_results,
    random_forest_eligible_thresholds
) = choose_threshold_for_minimum_recall(
    y_true=y_val,
    y_proba=y_val_proba_random_forest,
    min_recall=RECALL_TARGET,
    model_name=(
        "Random Forest validation selected"
    )
)

print("Selected Random Forest threshold:")
print(random_forest_selected_threshold)

pd.DataFrame([
    random_forest_val_default_results,
    random_forest_val_selected_results
])


Selected Random Forest threshold:
0.07043807295588024


,model,threshold,accuracy,precision,recall,specificity,false_positive_rate,false_negative_rate,f1,f2,...,auprc,brier_score,true_negative,false_positive,false_negative,true_positive,predicted_positive,predicted_negative,predicted_positive_rate,patients_flagged_per_true_readmission_found
0,Random Forest validation default,0.500000,0.910201,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,...,0.13961,0.080447,12741,0,1257,0,0,13998,0.000000,NaN
1,Random Forest validation selected,0.070438,0.394914,0.109051,0.800318,0.354917,0.645083,0.199682,0.191948,0.352908,...,0.13961,0.080447,4522,8219,251,1006,9225,4773,0.659023,9.16998


In [23]:
random_forest_eligible_thresholds.to_csv(
    OUTPUT_DIR
    / "random_forest_eligible_validation_thresholds.csv",
    index=False
)

random_forest_eligible_thresholds.head(20)


,threshold,recall,false_positive_rate,specificity
0,0.070438,0.800318,0.645083,0.354917
1,0.070431,0.800318,0.645161,0.354839
2,0.070431,0.800318,0.645240,0.354760
3,0.070428,0.800318,0.645318,0.354682
4,0.070427,0.800318,0.645397,0.354603
5,0.070426,0.800318,0.645475,0.354525
6,0.070423,0.800318,0.645554,0.354446
7,0.070422,0.800318,0.645632,0.354368
8,0.070420,0.800318,0.645711,0.354289
9,0.070419,0.800318,0.645789,0.354211


In [24]:
random_forest_threshold_sweep = threshold_sweep(
    y_true=y_val,
    y_proba=y_val_proba_random_forest,
    model_name="Random Forest validation"
)

random_forest_threshold_sweep.to_csv(
    OUTPUT_DIR
    / "random_forest_validation_threshold_sweep.csv",
    index=False
)

random_forest_threshold_sweep[
    [
        "threshold",
        "recall",
        "precision",
        "specificity",
        "false_positive_rate",
        "f1",
        "f2",
        "true_positive",
        "false_positive",
        "false_negative",
        "predicted_positive_rate"
    ]
]


,threshold,recall,precision,specificity,false_positive_rate,f1,f2,true_positive,false_positive,false_negative,predicted_positive_rate
0,0.01,1.000000,0.089799,0.000000,1.000000,0.164798,0.330337,1257,12741,0,1.000000
1,0.02,1.000000,0.089799,0.000000,1.000000,0.164798,0.330337,1257,12741,0,1.000000
2,0.03,1.000000,0.089799,0.000000,1.000000,0.164798,0.330337,1257,12741,0,1.000000
3,0.04,0.999204,0.089920,0.002276,0.997724,0.164992,0.330596,1256,12712,1,0.997857
4,0.05,0.987271,0.091987,0.038537,0.961463,0.168294,0.335061,1241,12250,16,0.963781
...,...,...,...,...,...,...,...,...,...,...,...
90,0.91,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0,0,1257,0.000000
91,0.92,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0,0,1257,0.000000
92,0.93,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0,0,1257,0.000000
93,0.94,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0,0,1257,0.000000


In [25]:
comparison_columns = [
    "model",
    "threshold",
    "auprc",
    "auroc",
    "brier_score",
    "accuracy",
    "recall",
    "precision",
    "specificity",
    "false_positive_rate",
    "false_negative_rate",
    "f1",
    "f2",
    "true_positive",
    "true_negative",
    "false_positive",
    "false_negative",
    "predicted_positive_rate",
    "patients_flagged_per_true_readmission_found"
]

random_forest_validation_comparison = pd.DataFrame([
    random_forest_val_default_results,
    random_forest_val_selected_results
])

random_forest_validation_comparison = (
    random_forest_validation_comparison[
        comparison_columns
    ]
)

random_forest_validation_comparison.to_csv(
    OUTPUT_DIR
    / "random_forest_validation_results.csv",
    index=False
)

random_forest_validation_comparison


,model,threshold,auprc,auroc,brier_score,accuracy,recall,precision,specificity,false_positive_rate,false_negative_rate,f1,f2,true_positive,true_negative,false_positive,false_negative,predicted_positive_rate,patients_flagged_per_true_readmission_found
0,Random Forest validation default,0.500000,0.13961,0.622744,0.080447,0.910201,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0,12741,0,1257,0.000000,NaN
1,Random Forest validation selected,0.070438,0.13961,0.622744,0.080447,0.394914,0.800318,0.109051,0.354917,0.645083,0.199682,0.191948,0.352908,1006,4522,8219,251,0.659023,9.16998


In [26]:
random_forest_val_selected_cm = (
    confusion_matrix_from_proba(
        y_true=y_val,
        y_proba=y_val_proba_random_forest,
        threshold=(
            random_forest_selected_threshold
        )
    )
)

random_forest_val_selected_cm.to_csv(
    OUTPUT_DIR
    / "random_forest_validation_confusion_matrix.csv"
)

random_forest_val_selected_cm


,Predicted not readmitted,Predicted readmitted
Actual not readmitted,4522,8219
Actual readmitted,251,1006


In [27]:
y_val_pred_random_forest = (
    y_val_proba_random_forest
    >= random_forest_selected_threshold
).astype(int)

print(
    classification_report(
        y_val,
        y_val_pred_random_forest,
        target_names=[
            "Not readmitted",
            "Readmitted"
        ],
        zero_division=0
    )
)


                precision    recall  f1-score   support

Not readmitted       0.95      0.35      0.52     12741
    Readmitted       0.11      0.80      0.19      1257

      accuracy                           0.39     13998
     macro avg       0.53      0.58      0.35     13998
  weighted avg       0.87      0.39      0.49     13998



## 10. Final evaluation on the untouched test set

In [28]:
y_test_proba_random_forest = (
    best_random_forest_model
    .predict_proba(X_test)[:, 1]
)

final_random_forest_test_results = (
    evaluate_predictions_from_proba(
        y_true=y_test,
        y_proba=y_test_proba_random_forest,
        threshold=(
            random_forest_selected_threshold
        ),
        model_name="Final Random Forest test"
    )
)

final_random_forest_test_results_df = pd.DataFrame([
    final_random_forest_test_results
])

final_random_forest_test_results_df


,model,threshold,accuracy,precision,recall,specificity,false_positive_rate,false_negative_rate,f1,f2,...,auprc,brier_score,true_negative,false_positive,false_negative,true_positive,predicted_positive,predicted_negative,predicted_positive_rate,patients_flagged_per_true_readmission_found
0,Final Random Forest test,0.070438,0.401414,0.111329,0.811456,0.360961,0.639039,0.188544,0.195796,0.359408,...,0.152591,0.079972,4599,8142,237,1020,9162,4836,0.654522,8.982353


In [29]:
final_random_forest_test_cm = (
    confusion_matrix_from_proba(
        y_true=y_test,
        y_proba=y_test_proba_random_forest,
        threshold=(
            random_forest_selected_threshold
        )
    )
)

final_random_forest_test_cm


,Predicted not readmitted,Predicted readmitted
Actual not readmitted,4599,8142
Actual readmitted,237,1020


In [30]:
y_test_pred_random_forest = (
    y_test_proba_random_forest
    >= random_forest_selected_threshold
).astype(int)

print(
    classification_report(
        y_test,
        y_test_pred_random_forest,
        target_names=[
            "Not readmitted",
            "Readmitted"
        ],
        zero_division=0
    )
)


                precision    recall  f1-score   support

Not readmitted       0.95      0.36      0.52     12741
    Readmitted       0.11      0.81      0.20      1257

      accuracy                           0.40     13998
     macro avg       0.53      0.59      0.36     13998
  weighted avg       0.88      0.40      0.49     13998



In [31]:
random_forest_validation_test_comparison = (
    pd.DataFrame([
        random_forest_val_selected_results,
        final_random_forest_test_results
    ])
)

random_forest_validation_test_comparison[
    [
        "model",
        "threshold",
        "auprc",
        "auroc",
        "recall",
        "precision",
        "specificity",
        "false_positive_rate",
        "f2",
        "predicted_positive_rate",
        "patients_flagged_per_true_readmission_found"
    ]
]


,model,threshold,auprc,auroc,recall,precision,specificity,false_positive_rate,f2,predicted_positive_rate,patients_flagged_per_true_readmission_found
0,Random Forest validation selected,0.070438,0.139610,0.622744,0.800318,0.109051,0.354917,0.645083,0.352908,0.659023,9.169980
1,Final Random Forest test,0.070438,0.152591,0.644518,0.811456,0.111329,0.360961,0.639039,0.359408,0.654522,8.982353


In [32]:
final_random_forest_test_results_df.to_csv(
    OUTPUT_DIR
    / "final_random_forest_test_metrics.csv",
    index=False
)

final_random_forest_test_cm.to_csv(
    OUTPUT_DIR
    / "final_random_forest_test_confusion_matrix.csv"
)

random_forest_validation_test_comparison.to_csv(
    OUTPUT_DIR
    / "random_forest_validation_test_comparison.csv",
    index=False
)

print("Final Random Forest results saved to:")
print(OUTPUT_DIR)


Final Random Forest results saved to:
/Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Model_Results/random_forest_optimisation


## 11. Random Forest feature importance

This is impurity-based feature importance averaged across the forest. It is useful for an initial inspection, but later permutation importance or SHAP should be preferred for a stronger interpretation.

In [33]:
fitted_preprocessor = (
    best_random_forest_model
    .named_steps["preprocess"]
)

transformed_feature_names = (
    fitted_preprocessor
    .get_feature_names_out()
)

print(
    "Number of transformed features:",
    len(transformed_feature_names)
)

transformed_feature_names[:20]


Number of transformed features: 46


array(['categorical__gender_Female', 'categorical__gender_Male',
       'categorical__race_group_AfricanAmerican',
       'categorical__race_group_Caucasian',
       'categorical__race_group_Missing', 'categorical__race_group_Other',
       'categorical__age_group_30-60', 'categorical__age_group_<=30',
       'categorical__age_group_>60',
       'categorical__admission_source_group_Emergency room',
       'categorical__admission_source_group_Other',
       'categorical__admission_source_group_Physician/clinic referral',
       'categorical__discharge_group_Home',
       'categorical__discharge_group_Other',
       'categorical__medical_specialty_group_Cardiology',
       'categorical__medical_specialty_group_Family/General Practice',
       'categorical__medical_specialty_group_Internal Medicine',
       'categorical__medical_specialty_group_Missing',
       'categorical__medical_specialty_group_Other',
       'categorical__medical_specialty_group_Surgery'], dtype=object)

In [34]:
random_forest_feature_importance = (
    pd.DataFrame({
        "feature": transformed_feature_names,
        "importance": (
            fitted_random_forest
            .feature_importances_
        )
    })
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)

random_forest_feature_importance.to_csv(
    OUTPUT_DIR
    / "random_forest_feature_importance.csv",
    index=False
)

random_forest_feature_importance.head(30)


,feature,importance
0,numeric__number_inpatient,0.167936
1,categorical__discharge_group_Home,0.104412
2,categorical__discharge_group_Other,0.090513
3,numeric__num_lab_procedures,0.075059
4,numeric__num_medications,0.069625
5,numeric__time_in_hospital,0.066286
6,numeric__number_diagnoses,0.041028
7,numeric__number_emergency,0.032672
8,numeric__num_procedures,0.030500
9,categorical__age_group_>60,0.023320


In [35]:
def identify_original_feature(
    transformed_feature_name,
    categorical_features,
    numeric_features
):
    """Map a transformed feature back to its original variable."""

    clean_name = (
        transformed_feature_name
        .replace("categorical__", "")
        .replace("numeric__", "")
    )

    for feature in categorical_features:
        if clean_name.startswith(
            f"{feature}_"
        ):
            return feature

    for feature in numeric_features:
        if clean_name == feature:
            return feature

    return clean_name


random_forest_feature_importance[
    "original_feature"
] = (
    random_forest_feature_importance[
        "feature"
    ]
    .apply(
        identify_original_feature,
        categorical_features=categorical_features,
        numeric_features=numeric_features
    )
)

random_forest_original_feature_importance = (
    random_forest_feature_importance
    .groupby(
        "original_feature",
        as_index=False
    )["importance"]
    .sum()
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)

random_forest_original_feature_importance.to_csv(
    OUTPUT_DIR
    / "random_forest_original_feature_importance.csv",
    index=False
)

random_forest_original_feature_importance


,original_feature,importance
0,discharge_group,0.194926
1,number_inpatient,0.167936
2,num_lab_procedures,0.075059
3,num_medications,0.069625
4,primary_diagnosis,0.069220
5,time_in_hospital,0.066286
6,age_group,0.047662
7,medical_specialty_group,0.045254
8,number_diagnoses,0.041028
9,number_emergency,0.032672


## 12. Save the selected settings and experiment summary

In [36]:
best_parameters_table = pd.DataFrame(
    [
        {
            "parameter": parameter,
            "selected_value": str(value)
        }
        for parameter, value
        in random_forest_search.best_params_.items()
    ]
)

best_parameters_table.to_csv(
    OUTPUT_DIR
    / "random_forest_best_parameters.csv",
    index=False
)

selected_threshold_table = pd.DataFrame([
    {
        "recall_target": RECALL_TARGET,
        "selected_validation_threshold": (
            random_forest_selected_threshold
        )
    }
])

selected_threshold_table.to_csv(
    OUTPUT_DIR
    / "random_forest_selected_threshold.csv",
    index=False
)

experiment_summary = pd.Series({
    "cross_validation_auprc": (
        random_forest_search.best_score_
    ),
    "selected_validation_threshold": (
        random_forest_selected_threshold
    ),
    "validation_recall": (
        random_forest_val_selected_results[
            "recall"
        ]
    ),
    "validation_precision": (
        random_forest_val_selected_results[
            "precision"
        ]
    ),
    "validation_false_positive_rate": (
        random_forest_val_selected_results[
            "false_positive_rate"
        ]
    ),
    "test_auprc": (
        final_random_forest_test_results[
            "auprc"
        ]
    ),
    "test_auroc": (
        final_random_forest_test_results[
            "auroc"
        ]
    ),
    "test_recall": (
        final_random_forest_test_results[
            "recall"
        ]
    ),
    "test_precision": (
        final_random_forest_test_results[
            "precision"
        ]
    ),
    "test_false_positive_rate": (
        final_random_forest_test_results[
            "false_positive_rate"
        ]
    ),
    "test_f2": (
        final_random_forest_test_results[
            "f2"
        ]
    ),
    "test_predicted_positive_rate": (
        final_random_forest_test_results[
            "predicted_positive_rate"
        ]
    ),
    "test_patients_flagged_per_true_readmission": (
        final_random_forest_test_results[
            "patients_flagged_per_true_readmission_found"
        ]
    ),
    "number_of_trees": len(
        fitted_random_forest.estimators_
    ),
    "mean_tree_depth": (
        individual_tree_depths.mean()
    ),
    "mean_leaf_count": (
        individual_tree_leaves.mean()
    )
})

experiment_summary.to_csv(
    OUTPUT_DIR
    / "random_forest_experiment_summary.csv",
    header=["value"]
)

experiment_summary


cross_validation_auprc                          0.152075
selected_validation_threshold                   0.070438
validation_recall                               0.800318
validation_precision                            0.109051
validation_false_positive_rate                  0.645083
test_auprc                                      0.152591
test_auroc                                      0.644518
test_recall                                     0.811456
test_precision                                  0.111329
test_false_positive_rate                        0.639039
test_f2                                         0.359408
test_predicted_positive_rate                    0.654522
test_patients_flagged_per_true_readmission      8.982353
number_of_trees                               500.000000
mean_tree_depth                                 8.000000
mean_leaf_count                               138.124000
dtype: float64